# 01 — Morphisms as the app: ingest → materialize → [UM]

This notebook **is the application**. Each code cell is a step of the app,
and the [UM] (`ewm-app::StateMachine`) runs them against the shared
`StateCache`.

Progression (slow, one morphism at a time):

1. **ingest** — the default padded n-gram morphism: SHA1 of the new HLLSet,
   three pointers per token, LUTs + hllsetLUT side effects;
2. **materialize** — LUT-first, ordered by default, `no_order` option;
3. **the [UM]** — the stateless driver over the shared state, cell by cell.

In [1]:
:dep ewm-app = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/ewm-app" }
:dep ewm-git = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/ewm-git" }
:dep context-tree = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/context-tree" }
:dep hllset-morphisms = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/hllset-morphisms" }
:dep hllset-contracts = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/hllset-contracts" }
:dep hllset-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-state-machine/crates/hllset-core" }

In [2]:
use ewm_app::{StateCache, StateMachine};
use hllset_contracts::BitAddress;
use hllset_morphisms::{
    gate, ingest, lut_scheme, materialize, materialize_no_order, Ingest, CHANNELS, CHANNEL_NAMES,
    CHANNEL_SEEDS, NG, NS, N_SEEDS, PAD,
};

println!("notebook app loaded: [UM] + morphisms");

notebook app loaded: [UM] + morphisms


---
## Morphism 1 — ingest

The default case: an ordered token collection in, the **SHA1 of the new
HLLSet** out. The side effects preserve the work — token LUTs and the
hllsetLUT with the three channel HLLSet keys.

In [3]:
let tokens: Vec<Vec<u8>> = ["the", "cat", "sat"]
    .iter()
    .map(|t| t.as_bytes().to_vec())
    .collect();

let ing = ingest(&tokens);
println!("default return — SHA1 of the new HLLSet:");
println!("  projection key: {}", ing.key);
for (ch, key) in ing.keys.iter().enumerate() {
    let lut = &ing.lut_names()[ch];
    println!(
        "  G{} key: {}  LUT = {} (scheme = {:?})",
        ch + 1,
        key,
        lut,
        lut_scheme(lut)
    );
}
println!("hllsetLUT: {} named entries", ing.hllset_lut.len());
for (ch, key) in ing.keys.iter().enumerate() {
    println!(
        "  {} = {}  (TH = {})",
        CHANNEL_NAMES[ch],
        key,
        ing.hllset_lut.th_named(CHANNEL_NAMES[ch], key)
    );
}

default return — SHA1 of the new HLLSet:


  projection key: h:a421491d50bfa3742c15672667a0bc6d9a3ed102


  G1 key: h:c1b894b666264b47d78bc061db7e63f17d746207  LUT = ng:G1 (scheme = Some("ng"))


  G2 key: h:1bbb1d9658640931592121398acae8627f8f03f0  LUT = ng:G2 (scheme = Some("ng"))


  G3 key: h:b7661bd623bca71dca2464c72d43fd66296706d8  LUT = ng:G3 (scheme = Some("ng"))


hllsetLUT: 3 named entries


  G1 = h:c1b894b666264b47d78bc061db7e63f17d746207  (TH = 1)


  G2 = h:1bbb1d9658640931592121398acae8627f8f03f0  (TH = 1)


  G3 = h:b7661bd623bca71dca2464c72d43fd66296706d8  (TH = 1)


()

Every real token has **three hashes pointing at it** — its 1-gram, 2-gram
and 3-gram (each channel uses its own seed). Let's check the three pointers
of `"cat"` inside `[the, cat, sat]`.

In [4]:
fn join_gram(parts: &[&[u8]]) -> Vec<u8> {
    if parts.len() == 1 {
        return parts[0].to_vec();
    }
    let mut out = Vec::new();
    for (i, p) in parts.iter().enumerate() {
        if i > 0 {
            out.push(0u8);
        }
        out.extend_from_slice(p);
    }
    out
}

let ing = ingest(["the", "cat", "sat"]);
for ch in 0..CHANNELS {
    let seed = CHANNEL_SEEDS[ch];
    let gram: Vec<u8> = match ch {
        0 => join_gram(&[b"cat"]),
        1 => join_gram(&[b"cat", b"sat"]),
        _ => join_gram(&[b"cat", b"sat", PAD]),
    };
    let addr = BitAddress::of_token_seeded(&gram, seed);
    let atom_set = ing.sketches[ch].has_bit(addr.reg(), addr.tz());
    let fiber_has_cat = ing.luts[ch].fiber(addr.bit()).contains(&b"cat".to_vec());
    println!(
        "channel {}: atom of {:?} set = {}, LUT fiber holds cat = {}",
        ch + 1,
        String::from_utf8_lossy(&gram).replace('\0', "|"),
        atom_set,
        fiber_has_cat
    );
}

channel 1: atom of "cat" set = true, LUT fiber holds cat = true


channel 2: atom of "cat|sat" set = true, LUT fiber holds cat = true


channel 3: atom of "cat|sat|<PAD>" set = true, LUT fiber holds cat = true


()

The same Gn names, the other bootstrap scheme: the **n-seed** path sets
the same G1/G2/G3 channels with seeded hashes. The HLLSet is scheme-agnostic;
the `ns` prefix on its keys says: use the n-seed LUTs — the plain set, no
order.

In [5]:
let mut ns = Ingest::new();
ns.ingest_tokens([&b"cat"[..], &b"sat"[..]]);

println!("n-seed projection key: {}", ns.key());
for (i, k) in ns.keys().iter().enumerate() {
    let lut = &ns.lut_names()[i];
    println!(
        "  G{} key: {}  LUT = {} (scheme = {:?})",
        i + 1,
        k,
        lut,
        lut_scheme(lut)
    );
}

// n-seed LUTs are orderless: materialize returns the plain set.
let plain = {
    let pairs: Vec<_> = (0..N_SEEDS)
        .map(|i| (ns.hllset(i), ns.lut(i)))
        .collect();
    hllset_morphisms::materialize::materialize(&pairs)
};
println!("n-seed materialize (set): {:?}", plain);
println!("scheme ng={} ns={}", NG, NS);

n-seed projection key: h:922918ed50371bdf8887c1a796444359f26f49a9


  G1 key: h:8d9eadcf4f5d60a488e40446c64da9972210f657  LUT = ns:G1 (scheme = Some("ns"))


  G2 key: h:74c7a49833aac10f32c737ba9d9cc6af81f9a359  LUT = ns:G2 (scheme = Some("ns"))


  G3 key: h:29eaeef9467f325a0346f040512cd6b900951d3f  LUT = ns:G3 (scheme = Some("ns"))


n-seed materialize (set): {[99, 97, 116], [115, 97, 116]}


scheme ng=ng ns=ns


**Gn are gates.** `G1 = G1_ng ∪ G1_ns` — one shared channel holds both
schemes' bits. A bit is anonymous: it does not remember which token or which
scheme set it (collisions are fine — the materializer disambiguates).
`gate(Gx, H) = Gx ∩ H` extracts the bits; materialization restores their
origin in the context of the specific HLLSet, via the chosen LUTs.

In [6]:
let ng2 = ingest(["cat", "sat"]);
// ns is still in scope from the previous cell.
let addr = BitAddress::of_token_seeded(b"cat", 0);
println!(
    "1-gram(cat) atom in n-gram G1: {}",
    ng2.sketches[0].has_bit(addr.reg(), addr.tz())
);
println!(
    "seed-0(cat) atom in n-seed G1: {}",
    ns.hllset(0).has_bit(addr.reg(), addr.tz())
);
let g1_shared = ng2.sketches[0].union(ns.hllset(0));
println!(
    "gate(G1, H_ng) == H_ng: {}",
    gate(&g1_shared, &ng2.sketches[0]).content_key() == ng2.sketches[0].content_key()
);
println!(
    "gate(G1, H_ns) == H_ns: {}",
    gate(&g1_shared, ns.hllset(0)).content_key() == ns.hllset(0).content_key()
);
let g2_shared = ng2.sketches[1].union(ns.hllset(1));
println!(
    "shared G2 bits = {}  (ng = {} + ns = {})",
    g2_shared.popcount(),
    ng2.sketches[1].popcount(),
    ns.hllset(1).popcount()
);
println!(
    "gate(G2, H_ng) == H_ng: {}",
    gate(&g2_shared, &ng2.sketches[1]).content_key() == ng2.sketches[1].content_key()
);
println!(
    "gate(G2, H_ns) == H_ns: {}",
    gate(&g2_shared, ns.hllset(1)).content_key() == ns.hllset(1).content_key()
);

1-gram(cat) atom in n-gram G1: true


seed-0(cat) atom in n-seed G1: true


gate(G1, H_ng) == H_ng: true


gate(G1, H_ns) == H_ns: true


shared G2 bits = 6  (ng = 4 + ns = 2)


gate(G2, H_ng) == H_ng: true


gate(G2, H_ns) == H_ns: true


---
## Morphism 2 — materialize

LUT-first over the three channels, **keeping every reference** (collided
bits restore all candidates — probabilistic restoration, no TF filtering).
**Ordered by default**: the 3-gram chain is a De Bruijn walk, and TF scores
the transitions — greedy decoding by default, `materialize_beam` keeps the
top-k hypotheses (like LLM beam search). `no_order` returns the plain set.

In [7]:
let tokens: Vec<Vec<u8>> = ["the", "cat", "sat", "on", "the", "mat"]
    .iter()
    .map(|t| t.as_bytes().to_vec())
    .collect();
let ing = ingest(&tokens);

let ordered = materialize(&ing);
println!("ordered restore == original: {}", ordered == tokens);

let set = materialize_no_order(&ing);
println!("no_order set: {:?}", set);

ordered restore == original: true


no_order set: {[99, 97, 116], [109, 97, 116], [111, 110], [115, 97, 116], [116, 104, 101]}


---
## The [UM] — this notebook is the app

The [UM] (`StateMachine`) is stateless: it owns only the store handle.
S(t) and H(t-1) live in the shared `StateCache`. Each cell below is one
turn of the app.

In [8]:
let mut um = StateMachine::new(ewm_git::MemoryStore::default());
let mut cache = StateCache::empty();

let turn1 = [10u32, 20, 30];
let out1 = um.run_turn(&mut cache, &turn1).expect("turn 1");
println!("commit: {}", out1.commit.as_ref().map(|c| c.to_string()).unwrap_or_else(|| "-".into()));
println!("tip:    {}", out1.head.as_ref().map(|h| h.to_string()).unwrap_or_else(|| "-".into()));
println!("S(t) leaves: {}", out1.tree.leaves().len());
println!("full_image: {:?}", out1.full_image);
{
    let v = out1.commit_view.as_ref().expect("root view");
    println!("D={} R={} N={}", v.departed.popcount(), v.retained.popcount(), v.new.popcount());
}

commit: d2ee26ac


tip:    d2ee26ac


S(t) leaves: 1


full_image: [[116, 105, 100, 49, 48], [116, 105, 100, 50, 48], [116, 105, 100, 51, 48]]


D=0 R=0 N=3


()

In [9]:
let turn2 = [20u32, 30, 40];
let out2 = um.run_turn(&mut cache, &turn2).expect("turn 2");
println!("commit: {}", out2.commit.as_ref().map(|c| c.to_string()).unwrap_or_else(|| "-".into()));
{
    let v = out2.commit_view.as_ref().expect("head view");
    println!("D={} R={} N={}", v.departed.popcount(), v.retained.popcount(), v.new.popcount());
}
println!(
    "tree D/R/N: added={:?} retained={:?} removed={:?}",
    out2.diff.added, out2.diff.retained, out2.diff.removed
);

commit: 70dfa612


D=0 R=3 N=1


tree D/R/N: added=["h:f97ac4a7b938f40d31bf300c497cd7bc11d8377b"] retained=["h:480a4fd737c5f205d41cc85bf0c973751a88e68e"] removed=[]


---
## The Boolean ring — context as a GF(2) window over originals

Each turn's original HLLSet (its seed-0 sketch) enters a **moving window**
in ingestion order, bounded by the cache (`RING_CAPACITY`). The window's
GF(2) basis is deterministic for that sequence; every new original reports
its **linear novelty** — the residual popcount it adds to the span, or
`in_span = true` when the turn is already expressible from what the context
has seen.

In [10]:
// The ring lives in the shared cache; run_turn pushes each original and
// reports the stats. `um` and `cache` are in scope from the [UM] cells.
let turn3 = [10u32, 20, 30]; // replay of turn 1 — already in the span
let out3 = um.run_turn(&mut cache, &turn3).expect("turn 3");
println!(
    "turn 3 (replay): residual={} in_span={} dim={}",
    out3.ring_stats.residual, out3.ring_stats.in_span, out3.ring_stats.dimension
);

let turn4 = [50u32, 60, 70]; // a fresh direction
let out4 = um.run_turn(&mut cache, &turn4).expect("turn 4");
println!(
    "turn 4 (fresh):  residual={} in_span={} dim={}",
    out4.ring_stats.residual, out4.ring_stats.in_span, out4.ring_stats.dimension
);

let snap = um.snapshot(&cache);
println!(
    "ring window: {}/{} originals, GF(2) dimension {}",
    snap.ring.window_len, snap.ring.capacity, snap.ring.dimension
);

turn 3 (replay): residual=0 in_span=true dim=2


turn 4 (fresh):  residual=3 in_span=false dim=3


ring window: 4/64 originals, GF(2) dimension 3


---
## The side-car loop — encodings in, ordered encodings out

The host line produces dense encodings; the [UM] consumes `tid{n}` ids.
The **encoder** is the only vocabulary-aware step: a shared codebook
quantizes the encodings, the [UM] does the rest, and ordered materialize
hands the ids back to the host.

In [11]:
use ewm_app::Encoder;
// Host encodings (f32 vectors) -> codebook quantizer -> tid ids -> [UM]
// -> HLLSets -> ordered materialize -> ids back to the host.
let encoder = ewm_app::CodebookEncoder::new(8, 16, 7);
let encodings: Vec<Vec<f32>> = vec![
    vec![1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    vec![0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    vec![1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
];

let ids = encoder.encode(&encodings);
println!("encodings -> ids: {:?}", ids);

let out = um
    .run_turn_encoded(&mut cache, &encoder, &encodings)
    .expect("side-car turn");
println!("ordered materialize hands back: {:?}", out.restored_ids);
println!(
    "ring: residual={} in_span={} dim={}",
    out.ring_stats.residual, out.ring_stats.in_span, out.ring_stats.dimension
);

encodings -> ids: [8, 7, 8]


ordered materialize hands back: [8, 7, 8]


ring: residual=2 in_span=false dim=4


---
## Recovery — pop, not rebuild

Crash: drop the [UM] **and** its cache. The persistent store is untouched; a
fresh [UM] and a restored cache resume from the tip.

In [12]:
let dir = std::env::temp_dir().join("um-notebook-recovery");
let _ = std::fs::remove_dir_all(&dir);

// First [UM] commits one turn, then "crashes" (goes out of scope).
{
    let mut um = StateMachine::new(ewm_git::LooseStore::new(&dir));
    let mut cache = StateCache::empty();
    um.run_turn(&mut cache, &[1u32, 2, 3]).expect("turn");
    println!("tip after first [UM]: {:?}", um.head().map(|h| h.to_string()));
}

// Fresh [UM] over the same store; fresh cache restored from the tip.
let mut um2 = StateMachine::open(ewm_git::LooseStore::new(&dir));
let mut cache2 = StateCache::restore(um2.repo());
println!("restored tip:        {:?}", um2.head().map(|h| h.to_string()));
println!("restored cache.tip:  {:?}", cache2.tip.as_ref().map(|h| h.to_string()));
println!("restored leaves:     {}", cache2.tree().leaves().len());

let resumed = um2.run_turn(&mut cache2, &[3u32, 4]).expect("resume");
println!("resumed commit:      {:?}", resumed.commit.as_ref().map(|c| c.to_string()));
let _ = std::fs::remove_dir_all(&dir);

tip after first [UM]: Some("420b333b")


restored tip:        Some("420b333b")


restored cache.tip:  Some("420b333b")


restored leaves:     1


resumed commit:      Some("6b0341eb")


---
## Summary

- **ingest** returns the SHA1 of the new HLLSet and preserves the work in
  LUTs + hllsetLUT;
- the **side-car loop** is wired into the [UM]: host encodings →
  `CodebookEncoder` → `run_turn_encoded` → ordered `restored_ids` back to
  the host — the encoder is the only vocabulary-aware step
- the **Boolean ring** turns the context into a GF(2) window over
  the original turn HLLSets: `ring_stats.residual` is the linear novelty of
  each turn, `ring.dimension` the context width (docs/BOOLRING.md)
- **materialize** restores the ordered token collection (or the plain set
  with `no_order`);
- the **[UM]** runs this notebook cell by cell as a stateless driver, with
  S(t) and H(t-1) outside it in the shareable `StateCache`;
- recovery is a fresh [UM] + `StateCache::restore` — a read of the tip, not
  a replay.